In [ ]:
from pathlib import Path

In [14]:
DIR = Path("basketball_data")
DIR.mkdir(exist_ok=True)

In [ ]:
from nba_api.stats.endpoints import (
  leaguedashplayerstats,
  leaguedashlineups,
)
SEASON = "2025-26"

In [9]:
def fetch_player_stats(season, measure_type="Base"):
  endpoints = leaguedashplayerstats.LeagueDashPlayerStats(
    season= season,
    season_type_all_star= "Regular Season",
    measure_type_detailed_defense= measure_type,
    per_mode_detailed= "PerGame",
    timeout= 60,
  )

  return endpoints.get_data_frames()[0]

player_base = fetch_player_stats(SEASON, "Base")

print("Rows", len(player_base))
print("Columns", len(player_base.columns))

player_base.head()

Rows 582
Columns 69


,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,...,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,FP_HIGH_SCORE_RANK,TEAM_COUNT
0,1630639,A.J. Lawson,A.J.,1610612761,TOR,25.0,24,12,12,0.500,...,439,450,445,286,481,284,38,467,487,1
1,1631260,AJ Green,AJ,1610612749,MIL,26.0,78,31,47,0.397,...,121,335,201,398,292,284,38,231,290,1
2,1642358,AJ Johnson,AJ,1610612742,DAL,21.0,48,9,39,0.188,...,520,456,475,339,502,284,38,501,500,2
3,203932,Aaron Gordon,Aaron,1610612743,DEN,30.0,36,27,9,0.750,...,274,66,88,12,113,114,38,111,124,1
4,1628988,Aaron Holiday,Aaron,1610612745,HOU,29.0,57,38,19,0.667,...,405,393,386,115,457,284,38,430,444,1


In [10]:
player_advanced = fetch_player_stats(SEASON, "Advanced")
player_advanced.head()

,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,...,E_PACE_RANK,PACE_RANK,sp_work_PACE_RANK,PIE_RANK,FGM_RANK,FGA_RANK,FGM_PG_RANK,FGA_PG_RANK,FG_PCT_RANK,TEAM_COUNT
0,1630639,A.J. Lawson,A.J.,1610612761,TOR,25.0,24,12,12,0.500,...,506,456,456,268,443,447,454,442,363,1
1,1631260,AJ Green,AJ,1610612749,MIL,26.0,78,31,47,0.397,...,415,467,467,497,149,129,237,211,403,1
2,1642358,AJ Johnson,AJ,1610612742,DAL,21.0,48,9,39,0.188,...,293,270,270,544,401,370,486,421,552,2
3,203932,Aaron Gordon,Aaron,1610612743,DEN,30.0,36,27,9,0.750,...,394,426,426,104,215,229,101,119,168,1
4,1628988,Aaron Holiday,Aaron,1610612745,HOU,29.0,57,38,19,0.667,...,451,445,445,468,332,316,402,385,434,1


In [11]:
advanced_extra_cols = [
    col for col in player_advanced.columns
    if col not in player_base.columns
    and col != "PLAYER_ID"
]

player_stats = player_base.merge(
    player_advanced[["PLAYER_ID"] + advanced_extra_cols],
    on="PLAYER_ID",
    how="left",
    validate="one_to_one",
)

player_stats["SEASON"] = SEASON

print("Combined shape:", player_stats.shape)
player_stats.head()

Combined shape: (582, 126)


,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,...,TS_PCT_RANK,USG_PCT_RANK,E_USG_PCT_RANK,E_PACE_RANK,PACE_RANK,sp_work_PACE_RANK,PIE_RANK,FGM_PG_RANK,FGA_PG_RANK,SEASON
0,1630639,A.J. Lawson,A.J.,1610612761,TOR,25.0,24,12,12,0.500,...,230,284,273,506,456,456,268,454,442,2025-26
1,1631260,AJ Green,AJ,1610612749,MIL,26.0,78,31,47,0.397,...,109,467,467,415,467,467,497,237,211,2025-26
2,1642358,AJ Johnson,AJ,1610612742,DAL,21.0,48,9,39,0.188,...,555,185,182,293,270,270,544,486,421,2025-26
3,203932,Aaron Gordon,Aaron,1610612743,DEN,30.0,36,27,9,0.750,...,123,133,134,394,426,426,104,101,119,2025-26
4,1628988,Aaron Holiday,Aaron,1610612745,HOU,29.0,57,38,19,0.667,...,272,350,342,451,445,445,468,402,385,2025-26


In [16]:
player_stats.to_csv(
    DIR / f"player_stats_{SEASON}.csv",
    index=False,
)

print("Saved player statistics.")

Saved player statistics.


In [20]:
print("Duplicate player IDs:", player_stats["PLAYER_ID"].duplicated().sum())

print("\nMissing values in most incomplete columns:")
display(
    player_stats.isna()
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

Duplicate player IDs: 0

Missing values in most incomplete columns:


PLAYER_ID            0
PLAYER_NAME          0
NICKNAME             0
TEAM_ID              0
TEAM_ABBREVIATION    0
AGE                  0
GP                   0
W                    0
L                    0
W_PCT                0
MIN                  0
FGM                  0
FGA                  0
FG_PCT               0
FG3M                 0
dtype: int64

In [19]:
print("\nPlayers with the most minutes per game:")
display(
    player_stats.sort_values("MIN", ascending=False)[
        ["PLAYER_NAME", "TEAM_ABBREVIATION", "GP", "MIN", "PTS", "AST"]
    ].head(10)
)


Players with the most minutes per game:


,PLAYER_NAME,TEAM_ABBREVIATION,GP,MIN,PTS,AST
554,Tyrese Maxey,PHI,70,38.0,28.3,6.6
21,Amen Thompson,HOU,79,37.4,18.3,5.3
333,Kevin Durant,HOU,78,36.4,26.0,4.8
526,Toby Okani,MEM,6,36.2,10.0,1.0
372,Luka Dončić,LAL,64,35.8,33.5,8.3
542,Trey Murphy III,NOP,66,35.5,21.5,3.8
242,Jamal Murray,DEN,75,35.4,25.4,7.1
234,Jalen Johnson,ATL,72,35.2,22.5,7.9
218,Jabari Smith Jr.,HOU,77,35.1,15.8,1.9
31,Anthony Edwards,MIN,61,35.0,28.8,3.7


In [21]:
def fetch_lineup_stats(season, measure_type="Advanced"):
    endpoint = leaguedashlineups.LeagueDashLineups(
        season=season,
        season_type_all_star="Regular Season",
        measure_type_detailed_defense=measure_type,
        per_mode_detailed="Per100Possessions",
        group_quantity=5,
        timeout=60,
    )

    return endpoint.get_data_frames()[0]


lineups = fetch_lineup_stats(SEASON)

print("Rows:", len(lineups))
print("Columns:", len(lineups.columns))

lineups.head()

Rows: 2000
Columns: 50


,GROUP_SET,GROUP_ID,GROUP_NAME,TEAM_ID,TEAM_ABBREVIATION,GP,W,L,W_PCT,MIN,...,AST_RATIO_RANK,OREB_PCT_RANK,DREB_PCT_RANK,REB_PCT_RANK,TM_TOV_PCT_RANK,EFG_PCT_RANK,TS_PCT_RANK,PACE_RANK,PIE_RANK,SUM_TIME_PLAYED
0,Lineups,-203497-203944-1628978-1630162-1630183-,R. Gobert - J. Randle - D. DiVincenzo - A. Edw...,1610612750,MIN,54,33,21,0.611,710.0,...,7586,7743,8283,7230,12224,6964,7423,12412,8142,2129110
1,Lineups,-1626157-1628384-1628404-1628969-1628973-,K. Towns - O. Anunoby - J. Hart - M. Bridges -...,1610612752,NYK,43,30,13,0.698,541.0,...,5320,9262,7136,7422,11547,7209,7622,12988,8378,1621680
2,Lineups,-202699-1629130-1630595-1631105-1641709-,T. Harris - D. Robinson - C. Cunningham - J. D...,1610612765,DET,42,29,13,0.690,536.0,...,5748,5963,8176,6797,11260,7608,8288,12331,7912,1609130
3,Lineups,-1628970-1630163-1631217-1641706-1642851-,M. Bridges - L. Ball - M. Diabaté - B. Miller ...,1610612766,CHA,49,34,15,0.694,509.0,...,5253,5920,8071,6505,11259,5687,6155,13741,6719,1527740
4,Lineups,-203468-1629638-1630168-1630552-1630700-,C. McCollum - N. Alexander-Walker - O. Okongwu...,1610612737,ATL,30,20,10,0.667,391.0,...,5653,7634,7134,7238,10540,7001,7439,11022,7338,1171820


In [22]:
print(lineups.columns.tolist())

['GROUP_SET', 'GROUP_ID', 'GROUP_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'E_OFF_RATING', 'OFF_RATING', 'E_DEF_RATING', 'DEF_RATING', 'E_NET_RATING', 'NET_RATING', 'AST_PCT', 'AST_TO', 'AST_RATIO', 'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'TM_TOV_PCT', 'EFG_PCT', 'TS_PCT', 'E_PACE', 'PACE', 'PACE_PER40', 'POSS', 'PIE', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'OFF_RATING_RANK', 'DEF_RATING_RANK', 'NET_RATING_RANK', 'AST_PCT_RANK', 'AST_TO_RANK', 'AST_RATIO_RANK', 'OREB_PCT_RANK', 'DREB_PCT_RANK', 'REB_PCT_RANK', 'TM_TOV_PCT_RANK', 'EFG_PCT_RANK', 'TS_PCT_RANK', 'PACE_RANK', 'PIE_RANK', 'SUM_TIME_PLAYED']


In [23]:
lineups["SEASON"] = SEASON

lineups.to_csv(
    DIR / f"lineups_{SEASON}.csv",
    index=False,
)

print("Saved lineup statistics.")

Saved lineup statistics.


In [24]:
if "MIN" in lineups.columns:
    display(
        lineups.sort_values("MIN", ascending=False)[
            [c for c in [
                "GROUP_NAME",
                "TEAM_ABBREVIATION",
                "GP",
                "MIN",
                "OFF_RATING",
                "DEF_RATING",
                "NET_RATING",
            ] if c in lineups.columns]
        ].head(20)
    )

,GROUP_NAME,TEAM_ABBREVIATION,GP,MIN,OFF_RATING,DEF_RATING,NET_RATING
0,R. Gobert - J. Randle - D. DiVincenzo - A. Edw...,MIN,54,710.0,118.8,110.9,7.9
1,K. Towns - O. Anunoby - J. Hart - M. Bridges -...,NYK,43,541.0,115.0,112.8,2.3
2,T. Harris - D. Robinson - C. Cunningham - J. D...,DET,42,536.0,121.4,109.9,11.4
3,M. Bridges - L. Ball - M. Diabaté - B. Miller ...,CHA,49,509.0,134.9,108.6,26.4
4,C. McCollum - N. Alexander-Walker - O. Okongwu...,ATL,30,391.0,123.1,102.8,20.3
5,K. Durant - J. Okogie - A. Sengun - J. Smith J...,HOU,41,377.0,121.8,111.5,10.3
6,A. Gordon - N. Jokić - J. Murray - C. Johnson ...,DEN,23,360.0,127.0,113.4,13.5
7,B. Ingram - J. Poeltl - R. Barrett - I. Quickl...,TOR,28,354.0,122.6,115.0,7.6
8,M. Porter Jr. - T. Mann - N. Claxton - N. Clow...,BKN,19,307.0,116.7,114.5,2.2
9,K. Durant - A. Sengun - J. Smith Jr. - T. Easo...,HOU,36,306.0,116.4,107.3,9.1
